In [42]:
%pip install scikit-learn pillow

Note: you may need to restart the kernel to use updated packages.


In [43]:
import random
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

In [47]:
OUTPUT_DIR = Path("/workspace/NeuroVision/dataset")
SEED = 42

CLASS_NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

In [44]:
pairs = []

for image_path in clean_images:
    matching_labels = [
        label for label in labels
        if label.stem == image_path.stem
    ]

    if len(matching_labels) == 1:
        pairs.append((image_path, matching_labels[0]))

print("Matched image-label pairs:", len(pairs))


Matched image-label pairs: 5247


In [45]:
class_ids = []

for image_path, label_path in pairs:
    with open(label_path, "r") as file:
        lines = file.readlines()

    if lines:
        class_id = int(lines[0].split()[0])
    else:
        class_id = -1

    class_ids.append(class_id)

In [ ]:

annotated_pairs = []
annotated_classes = []

empty_pairs = []

for pair, class_id in zip(pairs, class_ids):
    if class_id == -1:
        empty_pairs.append(pair)
    else:
        annotated_pairs.append(pair)
        annotated_classes.append(class_id)



train_pairs, temporary_pairs, train_classes, temporary_classes = train_test_split( annotated_pairs, annotated_classes, 
                                                                                  test_size=0.30, 
                                                                                  random_state=SEED, 
                                                                                  stratify=annotated_classes
)



valid_pairs, test_pairs = train_test_split( temporary_pairs, 
                                           test_size=0.50, 
                                           random_state=SEED, 
                                           stratify=temporary_classes
)



random.seed(SEED)
random.shuffle(empty_pairs)

for i, pair in enumerate(empty_pairs):
    if i % 3 == 0:
        train_pairs.append(pair)
    elif i % 3 == 1:
        valid_pairs.append(pair)
    else:
        test_pairs.append(pair)


# Display the results

splits = {
    "train": train_pairs,
    "valid": valid_pairs,
    "test": test_pairs
}

for split_name, split_pairs in splits.items():
    print(f"{split_name}: {len(split_pairs)} images")

train: 3673 images
valid: 787 images
test: 787 images


In [50]:
# Recreate the output folders to avoid mixing files from an older run.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split_name, split_pairs in splits.items():
    image_folder = OUTPUT_DIR / "images" / split_name
    label_folder = OUTPUT_DIR / "labels" / split_name

    image_folder.mkdir(parents=True, exist_ok=True)
    label_folder.mkdir(parents=True, exist_ok=True)

    for image_path, label_path in split_pairs:
        shutil.copy2(image_path, image_folder / image_path.name)
        shutil.copy2(label_path, label_folder / label_path.name)


In [51]:
for split_name in splits:
    image_folder = OUTPUT_DIR / "images" / split_name
    label_folder = OUTPUT_DIR / "labels" / split_name

    image_files = [
        file for file in image_folder.iterdir()
        if file.is_file()
    ]
    label_files = list(label_folder.glob("*.txt"))

    image_names = {file.stem for file in image_files}
    label_names = {file.stem for file in label_files}

    print(f"\n{split_name.upper()}")
    print("Images:", len(image_files))
    print("Annotation files:", len(label_files))
    print("Images without labels:", len(image_names - label_names))
    print("Labels without images:", len(label_names - image_names))


TRAIN
Images: 3673
Annotation files: 3673
Images without labels: 0
Labels without images: 0

VALID
Images: 787
Annotation files: 787
Images without labels: 0
Labels without images: 0

TEST
Images: 787
Annotation files: 787
Images without labels: 0
Labels without images: 0


In [52]:
yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/valid
test: images/test

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""

with open(OUTPUT_DIR / "data.yaml", "w") as file:
    file.write(yaml_content)

print("\nCreated:", OUTPUT_DIR / "data.yaml")



Created: /workspace/NeuroVision/dataset/data.yaml
